# Práctica 1 · Reto: verifiquen una cifra con el activo de su equipo

**Equipo:** *(escriban aquí el nombre del equipo y sus integrantes)*

Este notebook es **opcional** (nivel reto) y vive en el repositorio de **tu equipo**, en `notebooks/01_verificacion.ipynb`. Repite el ejercicio en parejas de la sesión 1, ahora con el activo que eligió su equipo.

**Entrega:** viernes 20 de noviembre de 2026, 23:59.

In [ ]:
# CELDA ESTÁNDAR DEL EQUIPO · Córrela siempre primero.
# En Colab descarga el repositorio de TU EQUIPO para poder usar src/finanzas.py.
import os
import subprocess
import sys

USUARIO = "tu-usuario"          # usuario de GitHub dueño del repositorio del equipo
REPO = "tu-repositorio"         # nombre del repositorio del equipo

if "google.colab" in sys.modules:
    if USUARIO == "tu-usuario":
        raise ValueError("Escribe arriba el usuario de GitHub y el nombre del repositorio de tu equipo.")
    if not os.path.isdir(f"/content/{REPO}"):
        subprocess.run(["git", "clone", "-q", f"https://github.com/{USUARIO}/{REPO}.git", f"/content/{REPO}"], check=True)
    else:   # ya estaba descargado: trae los últimos cambios del equipo
        subprocess.run(["git", "-C", f"/content/{REPO}", "pull", "-q"], check=False)
    os.chdir(f"/content/{REPO}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "numpy-financial"], check=True)
else:
    while not os.path.isdir("src") and os.getcwd() != os.path.dirname(os.getcwd()):
        os.chdir("..")
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print("Carpeta de trabajo:", os.getcwd())

## 1 · Los datos de su activo

Escriban el *ticker* de su activo como aparece en Yahoo Finance. Las emisoras de la BMV terminan en `.MX` (por ejemplo `GFNORTEO.MX`). Las fechas quedan fijas para que el resultado no dependa del día en que corran el notebook.

La función de abajo ya está hecha; solo córranla. La primera vez descarga los precios y los guarda en `data/raw`; las siguientes veces lee ese archivo, así todos en el equipo trabajan con los mismos datos. La descarga la estudiaremos a fondo en la sesión 4.

In [ ]:
import numpy as np
import pandas as pd

TICKER = "WALMEX.MX"                      # cambien esto por el activo de su equipo
INICIO, FIN = "2016-01-01", "2025-12-31"  # fechas fijas


def cargar_precios_ajustados(ticker, inicio, fin):
    """Devuelve el precio de cierre ajustado de un activo como Serie con índice de fechas.

    Si ya existe data/raw/<ticker>.csv lo lee; si no, descarga de Yahoo Finance y lo guarda.
    """
    archivo = os.path.join("data", "raw", ticker.split(".")[0].lower() + ".csv")
    if os.path.exists(archivo):
        tabla = pd.read_csv(archivo, parse_dates=["fecha"], index_col="fecha")
        return tabla["cierre_ajustado"]

    import yfinance as yf

    descarga = yf.download(ticker, start=inicio, end=fin, auto_adjust=True, progress=False)
    cierre = descarga["Close"]
    if isinstance(cierre, pd.DataFrame):   # algunas versiones devuelven una tabla de una columna
        cierre = cierre.iloc[:, 0]
    cierre = cierre.dropna().rename("cierre_ajustado")
    cierre.index.name = "fecha"
    if cierre.empty:
        raise ValueError(f"Yahoo Finance no devolvió datos para {ticker}. Revisen el ticker.")
    os.makedirs(os.path.dirname(archivo), exist_ok=True)
    cierre.to_csv(archivo)
    return cierre


precios = cargar_precios_ajustados(TICKER, INICIO, FIN)
print(TICKER, "·", precios.index.min().date(), "a", precios.index.max().date(), "·", len(precios), "días")

## 2 · La cifra del LLM

Pregunten a un LLM por una cifra de su activo que se pueda calcular con precios. Sugerencia:

> ¿Cuál fue la volatilidad anualizada de la acción de *(su emisora)* en 2025?

Escriban la pregunta exacta, el nombre del LLM y la cifra que les dio.

In [ ]:
LLM_USADO = ""            # por ejemplo "ChatGPT", "Gemini", "Claude"
PREGUNTA = ""             # la pregunta exacta que hicieron
CIFRA_LLM = None          # la cifra en decimales, por ejemplo 0.22

## 3 · La cifra con datos

Calculen la volatilidad anualizada de 2025: desviación estándar de los rendimientos logarítmicos diarios de 2025, multiplicada por $\sqrt{252}$. Guárdenla en `volatilidad_2025`.

In [ ]:
# Tu código aquí
print(f"Volatilidad anualizada de {TICKER} en 2025, con datos: {volatilidad_2025:.2%}")
if CIFRA_LLM is not None:
    print(f"Cifra de {LLM_USADO}: {CIFRA_LLM:.2%} · diferencia: {CIFRA_LLM - volatilidad_2025:+.2%}")

In [ ]:
# Comprobación
assert 0.05 < volatilidad_2025 < 1.5, "Revisen el cálculo: una volatilidad anualizada suele estar entre 5% y 150%."
print("El cálculo es razonable.")

## 4 · Conclusión

En tres o cuatro renglones: ¿coincidió la cifra del LLM con la de los datos? Si no, ¿a qué creen que se debe la diferencia? ¿Qué le faltó decir al LLM para que su cifra se pudiera comprobar?

*Escribe aquí tu respuesta.*

## Antes de entregar

- [ ] **Entorno de ejecución → Reiniciar sesión y ejecutar todo** corre sin errores.
- [ ] El archivo de `data/raw` quedó guardado en el repositorio del equipo.
- [ ] Declararon en el README qué LLM usaron y para qué.